## Ex:2 Data Wrangling and Transformation 
### Objective

To perform data wrangling and transformation on a dataset using Python and Pandas by handling missing values, removing duplicates, correcting data types, filtering data, and transforming variables into a suitable format for data analysis and machine learning.



##  Dataset Description

The dataset contains information about **student academic performance, educational background, MBA specialization, and placement salary**.

###  Dataset Attributes

| Column | Description |
|:---|:---|
| `sl_no` | Serial number / student identifier |
| `gender` | Gender of the student |
| `hsc_p` | Higher Secondary / 12th percentage |
| `hsc_s` | Higher Secondary stream |
| `degree_p` | Undergraduate degree percentage |
| `degree_t` | Undergraduate degree type |
| `etest_p` | Employability / entrance test percentage |
| `specialisation` | MBA specialization |
| `mba_p` | MBA percentage |
| `salary` | Salary offered after placement |

---

## Experiment Question

 **Using the given student placement dataset, perform data wrangling and transformation by handling missing values, scaling numerical features, detecting and treating outliers, encoding categorical variables, and generating a final model-ready dataset.**




Data wrangling and transformation is the process of converting raw student placement data into a clean, consistent, and machine-learning-ready dataset. In this experiment, the given student placement dataset is first loaded into a Pandas DataFrame and explored by examining its rows, columns, data types, and descriptive statistics. Missing values are then identified and handled by removing records with missing `salary` values and replacing missing values in `hsc_p`, `degree_p`, and `etest_p` with their respective mean values. The numerical attributes such as `hsc_p`, `degree_p`, `etest_p`, and `salary` are transformed using feature-scaling techniques such as `StandardScaler` and `MinMaxScaler` to bring the variables into suitable numerical ranges. The dataset is then divided into input features (`X`) and the target variable (`Y`), where `salary` is considered the target. The preprocessed data is saved as `Pre.csv` for further processing. Next, possible outliers in the `salary` attribute are identified using a boxplot and statistically detected using the Z-score method, where values with an absolute Z-score greater than 3 are considered potential outliers. Outliers are also treated using the capping and flooring method by calculating the 5th and 95th percentiles and replacing values outside these limits with the corresponding boundary values. The salary distribution before and after outlier treatment is then compared using visualization. Since the dataset also contains categorical attributes such as `gender`, `hsc_s`, `degree_t`, and `specialisation`, these variables are converted into numerical representations using `LabelEncoder` and One-Hot Encoding. Finally, the completely transformed dataset is verified for missing values, data types, dimensions, and numerical representation, and the resulting model-ready dataset is saved as `Final.csv`. Thus, the experiment demonstrates the complete workflow of preparing real-world tabular data for data analysis and machine-learning applications.

## Step 1: Start by importing the necessary Python libraries for data preprocessing.


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from scipy.stats import zscore
from scipy import stats
from sklearn.preprocessing import LabelEncoder

## Step 2: Load the placement dataset into a Pandas Dataframe.

In [9]:
df=pd.read_csv("data.csv")
df.info()
df.shape
df.head()
df.tail()
df.sample(5)
df.describe()
df.loc[0]
df.iloc[0]
df[0:2]
df["degree_p"]

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           215 non-null    int64  
 1   gender          215 non-null    str    
 2   hsc_p           210 non-null    float64
 3   hsc_s           215 non-null    str    
 4   degree_p        213 non-null    float64
 5   degree_t        215 non-null    str    
 6   etest_p         211 non-null    float64
 7   specialisation  215 non-null    str    
 8   mba_p           214 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 21.9 KB


0      58.00
1      77.48
2      64.00
3        NaN
4      73.30
       ...  
210    77.60
211    72.00
212    73.00
213    58.00
214    53.00
Name: degree_p, Length: 215, dtype: float64

## Step 3:Take a quick look at the data to understand its structure and identify any missing values or anomalies.

In [10]:
df.isnull().sum()



sl_no              0
gender             0
hsc_p              5
hsc_s              0
degree_p           2
degree_t           0
etest_p            4
specialisation     0
mba_p              1
salary            67
dtype: int64

#### The method isnull() checks each element in the DataFrame (or Series) to see if it is NaN (Not a Number) or None (missing value).
It returns a DataFrame (or Series) of the same shape as the input, with Boolean values:
#### True: The value is null (NaN or None).
#### False: The value is not null.

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           215 non-null    int64  
 1   gender          215 non-null    str    
 2   hsc_p           210 non-null    float64
 3   hsc_s           215 non-null    str    
 4   degree_p        213 non-null    float64
 5   degree_t        215 non-null    str    
 6   etest_p         211 non-null    float64
 7   specialisation  215 non-null    str    
 8   mba_p           214 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 21.9 KB


## Step 4: Handle Missing Data
### Option 1: If the dataset is large and only a small percentage of data is missing, you can remove rows with missing values using dropna(subset,inplace)


In [15]:
df.dropna(subset=["salary"],inplace=True)
df.isnull().sum()

sl_no             0
gender            0
hsc_p             2
hsc_s             0
degree_p          1
degree_t          0
etest_p           2
specialisation    0
mba_p             0
salary            0
dtype: int64

### Option 2:If removing data isn't ideal, you can impute (df.[""].fillna(df[""].mean(),inplace)) missing values using methods like mean, median, or most frequent.

In [17]:
df["hsc_p"]=df["hsc_p"].fillna(df["hsc_p"].mean())
df["degree_p"]=df["degree_p"].fillna(df["degree_p"].mean())
df["etest_p"]=df["etest_p"].fillna(df["etest_p"].mean())

In [18]:
df.isnull().sum()

sl_no             0
gender            0
hsc_p             0
hsc_s             0
degree_p          0
degree_t          0
etest_p           0
specialisation    0
mba_p             0
salary            0
dtype: int64

## Step 5: Feature Scaling
Feature scaling is the process of converting numerical features to a similar scale so that one feature does not dominate another simply because it has larger numerical values.

<img src="https://i.postimg.cc/G21gMYnF/f.png" alt="Image Description" width="500">









## Option 1( StandardScaler): This method scales the data to have a mean of 0 and a standard deviation of 1.


In [21]:
c=["hsc_p","degree_p","etest_p","salary"]
s1=StandardScaler()
df[c]=s1.fit_transform(df[c])
df.head()

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,2.265997e+00,Commerce,-1.652293,Sci&Tech,-1.328518,Mkt&HR,58.80,-0.200292
1,2,M,8.987875e-01,Science,1.346845,Sci&Tech,0.978332,Mkt&Fin,66.28,-0.951839
2,3,M,1.533482e-15,Arts,-0.728534,Comm&Mgmt,0.136149,Mkt&Fin,57.80,-0.415019
4,5,M,3.883770e-01,Commerce,0.703293,Comm&Mgmt,1.732635,Mkt&Fin,55.50,1.463849
7,8,M,-6.475513e-01,Science,-0.420614,Sci&Tech,-0.449718,Mkt&Fin,62.14,-0.393547


#### Option 2:This method scales the data to a fixed range, usually between 0 and 1. 
###  MinMaxScaler()

In [26]:
c=["hsc_p","degree_p","etest_p","salary"]
s2=MinMaxScaler()
df[c]=s1.fit_transform(df[c])
df.head()

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,0.857051,Commerce,0.057143,Sci&Tech,0.104167,Mkt&HR,58.80,0.094595
1,2,M,0.586729,Science,0.613714,Sci&Tech,0.760417,Mkt&Fin,66.28,0.000000
2,3,M,0.409023,Arts,0.228571,Comm&Mgmt,0.520833,Mkt&Fin,57.80,0.067568
4,5,M,0.485812,Commerce,0.494286,Comm&Mgmt,0.975000,Mkt&Fin,55.50,0.304054
7,8,M,0.280990,Science,0.285714,Sci&Tech,0.354167,Mkt&Fin,62.14,0.070270


## Step 6  Option 1: Identifying Outliers Using Z-Scores
The value of 3 in the context of Z-scores is often used as a threshold to identify outliers in a dataset. A Z-score represents how many standard deviations a data point is away from the mean of the dataset. Specifically:

A Z-score of 0 means the data point is exactly at the mean.
A Z-score of 1 means the data point is one standard deviation above the mean, and so on.
A Z-score of 3 corresponds to a data point being 3 standard deviations away from the mean. For a normal distribution, about 99.7% of the data points fall within 3 standard deviations of the mean (according to the 68-95-99.7 rule, which describes the spread of data in a normal distribution). Therefore, points with Z-scores greater than 3 or less than -3 are considered unusually far from the mean and are often flagged as outliers.

This threshold (Z > 3 or Z < -3) is commonly used in many statistical applications because it captures the extreme values that are rare in a normal distribution, which are typically considered to be outliers. However, the choice of threshold can vary depending on the specific application and the nature of the data.



[![Chat-GPT-Image-Aug-20-2026-08-06-05-PM.png](https://i.postimg.cc/2SFTtcXL/Chat-GPT-Image-Aug-20-2026-08-06-05-PM.png)](https://postimg.cc/4Yyz75GX)

In [30]:
columns_to_check=["salary"]
z_score=stats.zscore(df[columns_to_check])
u=(z_score>3)
l=(z_score<-3)
indx=u | l
clean_df=df[~indx]
clean_df.info()

<class 'pandas.DataFrame'>
Index: 145 entries, 0 to 213
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           145 non-null    int64  
 1   gender          145 non-null    str    
 2   hsc_p           145 non-null    float64
 3   hsc_s           145 non-null    str    
 4   degree_p        145 non-null    float64
 5   degree_t        145 non-null    str    
 6   etest_p         145 non-null    float64
 7   specialisation  145 non-null    str    
 8   mba_p           145 non-null    float64
 9   salary          145 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 15.9 KB


### Option 2:  Capping and Flooring Outliers
Capping and flooring is an outlier-treatment technique where extreme values are replaced with predefined boundary values instead of deleting the records.

In [33]:
l1=df["salary"].quantile(0.05)
ul=df["salary"].quantile(0.95)
df_capped=df.copy()
df_capped["salary"].clip(l1,ul)
df_capped.info()
df.head()

<class 'pandas.DataFrame'>
Index: 148 entries, 0 to 213
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           148 non-null    int64  
 1   gender          148 non-null    str    
 2   hsc_p           148 non-null    float64
 3   hsc_s           148 non-null    str    
 4   degree_p        148 non-null    float64
 5   degree_t        148 non-null    str    
 6   etest_p         148 non-null    float64
 7   specialisation  148 non-null    str    
 8   mba_p           148 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 16.2 KB


,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,M,0.857051,Commerce,0.057143,Sci&Tech,0.104167,Mkt&HR,58.80,0.094595
1,2,M,0.586729,Science,0.613714,Sci&Tech,0.760417,Mkt&Fin,66.28,0.000000
2,3,M,0.409023,Arts,0.228571,Comm&Mgmt,0.520833,Mkt&Fin,57.80,0.067568
4,5,M,0.485812,Commerce,0.494286,Comm&Mgmt,0.975000,Mkt&Fin,55.50,0.304054
7,8,M,0.280990,Science,0.285714,Sci&Tech,0.354167,Mkt&Fin,62.14,0.070270


## Step 7: Convert categorical variables into numerical format using LabelEncoder ().
[![Picture1.png](https://i.postimg.cc/yNpNvnVd/Picture1.png)](https://postimg.cc/zLW5fCHZ)




## Convert categorical variables into numerical format using one hot encoder

[![Picture2.png](https://i.postimg.cc/gcZ0Hv1J/Picture2.png)](https://postimg.cc/HjTHp7YD)

In [34]:
L1=LabelEncoder()
df["gender"]=L1.fit_transform(df["gender"])
df["hsc_s"]=L1.fit_transform(df["hsc_s"])
df["degree_t"]=L1.fit_transform(df["degree_t"])
df["specialisation"]=L1.fit_transform(df["specialisation"])
df.head()

,sl_no,gender,hsc_p,hsc_s,degree_p,degree_t,etest_p,specialisation,mba_p,salary
0,1,1,0.857051,1,0.057143,2,0.104167,1,58.80,0.094595
1,2,1,0.586729,2,0.613714,2,0.760417,0,66.28,0.000000
2,3,1,0.409023,0,0.228571,0,0.520833,0,57.80,0.067568
4,5,1,0.485812,1,0.494286,0,0.975000,0,55.50,0.304054
7,8,1,0.280990,2,0.285714,2,0.354167,0,62.14,0.070270


In [38]:
c=["gender"]
one_hot_encoded_data=pd.get_dummies(df,columns=c)
one_hot_encoded_data.to_csv('clean.csv',index=False)
one_hot_encoded_data.head()
Y=df['salary'].copy()

# Remove target and ID from features

X= df.drop(columns=['salary','sl_no']).copy()
df.info()

<class 'pandas.DataFrame'>
Index: 148 entries, 0 to 213
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           148 non-null    int64  
 1   gender          148 non-null    int64  
 2   hsc_p           148 non-null    float64
 3   hsc_s           148 non-null    int64  
 4   degree_p        148 non-null    float64
 5   degree_t        148 non-null    int64  
 6   etest_p         148 non-null    float64
 7   specialisation  148 non-null    int64  
 8   mba_p           148 non-null    float64
 9   salary          148 non-null    float64
dtypes: float64(5), int64(5)
memory usage: 12.7 KB


# Exercise: Data Cleaning and Transformation – Automobile Dataset

## Step 1: Load and Explore the Dataset

### 1. Load the Dataset
- Import Pandas and load the Automobile dataset.
- Display the first 10 rows.
- Display the shape of the dataset.

### 2. Explore the Dataset
- Display the column names.
- Display the data types.
- Generate descriptive statistics.
- Identify numerical and categorical columns.
- Display unique values in categorical columns.

## Step 2: Data Cleaning

### 3. Check Missing Values
- Check for missing values in each column.
- Display the number and percentage of missing values.

### 4. Handle Missing Values
- Replace missing numerical values using mean or median.
- Replace missing categorical values using mode.
- Verify that no missing values remain.

### 5. Remove Duplicate Records
- Check for duplicate rows.
- Display the number of duplicate records.
- Remove duplicate records.
- Verify the result.

### 6. Clean the `horsepower` Column
- Identify non-numeric values such as `?`.
- Replace `?` with `NaN`.
- Convert `horsepower` to numeric.
- Handle the resulting missing values.

## Step 3: Data Transformation

### 7. Transform the `origin` Column
- Display the unique values in `origin`.
- Convert the values into meaningful labels:
  - `1` → `usa`
  - `2` → `europe`
  - `3` → `japan`

### 8. Create `weight_kg`
- Create a new column `weight_kg`.
- Convert weight from pounds to kilograms.

  `weight_kg = weight × 0.453592`

### 9. Create `mpg_category`
Create a new column based on `mpg`:
- `< 20` → `Low`
- `20–29` → `Medium`
- `≥ 30` → `High`

### 10. Create `vehicle_age`
- Create a new column `vehicle_age`.
- Assume the current year is 2026.

  `vehicle_age = 2026 - model_year`

### 11. Rename Columns
Rename:
- `mpg` → `miles_per_gallon`
- `horsepower` → `hp`
- `weight` → `weight_lbs`
- `model_year` → `year`

### 12. Filter the Data
Display vehicles:
- With `mpg > 30`
- With `horsepower > 150`
- With `cylinders >= 6`
- Manufactured after 1980
- Originating from `usa`

## Step 4: Encoding Categorical Data

### 13. Label Encoding
- Apply `LabelEncoder` to the `origin` column.
- Create a new column `origin_encoded`.
- Display the original and encoded values.
- Display the category-to-label mapping.

### 14. One-Hot Encoding
- Apply One-Hot Encoding to the `origin` column.
- Compare Label Encoding and One-Hot Encoding.
- Which encoding method is more appropriate for `origin`? Explain why.

## Step 5: Outlier Detection

### 15. Identify Outliers Using Z-Scores
- Calculate the Z-score for the numerical features.
- Identify observations with `|Z-score| > 3` as outliers.
- Count the outliers in each numerical column.
- Display the rows containing outliers.
- Decide whether the outliers should be removed or retained.

## Step 6: Feature Scaling

### 16. Standardization Using StandardScaler
- Select the numerical features.
- Apply `StandardScaler`.
- Display the standardized values.
- Verify that the features have approximately mean `0` and standard deviation `1`.

## Step 7: Normalization

### 17. Normalization Using MinMaxScaler
- Apply `MinMaxScaler` to the numerical features.
- Transform the features to the range `[0, 1]`.
- Display the normalized values.
- Compare **Standardization** and **Normalization**.
- Explain when each scaling method is appropriate.

## Step 8: Create Features and Target

### 18. Create X and Y Variables
- Select the appropriate input features as **X (independent variables)**.
- Select `mpg` as **Y (target variable)**.
- Display the shape of `X` and `Y`.
- Save `X` and `Y` into `automobile_X_Y.csv`.
- Load the CSV file again and display the first 5 rows.

## Step 9: Save the Final Dataset

### 19. Save the Preprocessed Dataset
- Combine the processed features and target variable.
- Display the final dataset.
- Check for missing values.
- Save the final dataset as `automobile_preprocessed.csv`.

In [9]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy import stats


# ==========================================
# 1. LOAD DATASET
# ==========================================

df = pd.read_csv("automobile.csv")

print(df.head(10))
print("Shape:", df.shape)


# ==========================================
# 2. EXPLORE DATASET
# ==========================================

print("\nColumn Names:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

print("\nDescriptive Statistics:")
print(df.describe())

print("\nUnique Categorical Values:")

for col in df.select_dtypes(include="object").columns:
    print(col, df[col].unique())


# ==========================================
# 3. CHECK MISSING VALUES
# ==========================================

missing_data = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage":
        (df.isnull().sum() / len(df)) * 100
})

print("\nMissing Values:")
print(missing_data)


# ==========================================
# 4. CLEAN HORSEPOWER
# ==========================================

if "horsepower" in df.columns:

    df["horsepower"] = df["horsepower"].replace(
        "?", np.nan
    )

    df["horsepower"] = pd.to_numeric(
        df["horsepower"],
        errors="coerce"
    )


# ==========================================
# 5. HANDLE MISSING VALUES
# ==========================================

numerical_columns = df.select_dtypes(
    include=np.number
).columns

for col in numerical_columns:
    df[col] = df[col].fillna(
        df[col].median()
    )


categorical_columns = df.select_dtypes(
    include="object"
).columns

for col in categorical_columns:
    df[col] = df[col].fillna(
        df[col].mode()[0]
    )


print("\nMissing values after cleaning:")
print(df.isnull().sum())


# ==========================================
# 6. REMOVE DUPLICATES
# ==========================================

print(
    "\nDuplicate rows:",
    df.duplicated().sum()
)

df = df.drop_duplicates()

print(
    "Duplicates after removal:",
    df.duplicated().sum()
)


# ==========================================
# 7. TRANSFORM ORIGIN
# ==========================================

df["origin"] = df["origin"].replace({
    1: "usa",
    2: "europe",
    3: "japan"
})


# ==========================================
# 8. CREATE WEIGHT_KG
# ==========================================

df["weight_kg"] = (
    df["weight"] * 0.453592
)


# ==========================================
# 9. CREATE MPG CATEGORY
# ==========================================

def mpg_category(mpg):

    if mpg < 20:
        return "Low"

    elif mpg < 30:
        return "Medium"

    else:
        return "High"


df["mpg_category"] = (
    df["mpg"].apply(mpg_category)
)


# ==========================================
# 10. CREATE VEHICLE AGE
# ==========================================

df["vehicle_age"] = (
    2026 - df["model_year"]
)


# ==========================================
# 11. LABEL ENCODING
# ==========================================

le = LabelEncoder()

df["origin_encoded"] = (
    le.fit_transform(df["origin"])
)

print("\nLabel Encoding Mapping:")

mapping = dict(
    zip(
        le.classes_,
        le.transform(le.classes_)
    )
)

print(mapping)


# ==========================================
# 12. ONE-HOT ENCODING
# ==========================================

one_hot = pd.get_dummies(
    df["origin"],
    prefix="origin"
)

df = pd.concat(
    [df, one_hot],
    axis=1
)


# ==========================================
# 13. OUTLIER DETECTION
# ==========================================

numerical_columns = df.select_dtypes(
    include=np.number
).columns

z_scores = np.abs(
    stats.zscore(df[numerical_columns])
)

outlier_mask = z_scores > 3

outlier_counts = pd.Series(
    outlier_mask.sum(axis=0),
    index=numerical_columns
)

print("\nOutlier Count:")
print(outlier_counts)

outlier_rows = df[
    outlier_mask.any(axis=1)
]

print("\nOutlier Rows:")
print(outlier_rows)


# ==========================================
# 14. STANDARDIZATION
# ==========================================

features = [
    "cylinders",
    "displacement",
    "horsepower",
    "weight",
    "acceleration",
    "model_year"
]

standard_scaler = StandardScaler()

df_standardized = df.copy()

df_standardized[features] = (
    standard_scaler.fit_transform(
        df_standardized[features]
    )
)

print("\nStandardized Data:")
print(
    df_standardized[features].head()
)


# ==========================================
# 15. NORMALIZATION
# ==========================================

minmax_scaler = MinMaxScaler()

df_normalized = df.copy()

df_normalized[features] = (
    minmax_scaler.fit_transform(
        df_normalized[features]
    )
)

print("\nNormalized Data:")
print(
    df_normalized[features].head()
)


# ==========================================
# 16. RENAME COLUMNS
# ==========================================

df.rename(columns={
    "mpg": "miles_per_gallon",
    "horsepower": "hp",
    "weight": "weight_lbs",
    "model_year": "year"
}, inplace=True)


# ==========================================
# 17. CREATE X AND Y
# ==========================================

Y = df["miles_per_gallon"]

X = df.drop(
    columns=["miles_per_gallon"]
)

print("\nX Shape:", X.shape)
print("Y Shape:", Y.shape)


# ==========================================
# 18. SAVE X AND Y
# ==========================================

automobile_X_Y = pd.concat(
    [X, Y],
    axis=1
)

automobile_X_Y.to_csv(
    "automobile_X_Y.csv",
    index=False
)


# ==========================================
# 19. SAVE FINAL DATASET
# ==========================================

df.to_csv(
    "automobile_preprocessed.csv",
    index=False
)

print(
    "\nFinal dataset saved successfully!"
)

print("\nFinal Dataset:")
print(df.head())

print("\nFinal Missing Values:")
print(df.isnull().sum())

                        name   mpg  cylinders  displacement  horsepower  \
0  chevrolet chevelle malibu  18.0        8.0         307.0       130.0   
1          buick skylark 320  15.0        8.0         350.0       165.0   
2         plymouth satellite  18.0        8.0         318.0       150.0   
3              amc rebel sst  16.0        8.0         304.0       150.0   
4                ford torino  17.0        NaN         302.0       140.0   
5           ford galaxie 500  15.0        8.0         429.0       198.0   
6           chevrolet impala  14.0        8.0         454.0       220.0   
7          plymouth fury iii  14.0        8.0         440.0       215.0   
8           pontiac catalina  14.0        8.0         455.0         NaN   
9         amc ambassador dpl  15.0        8.0         390.0       190.0   

   weight  acceleration  model_year origin  
0  3504.0          12.0          70    usa  
1  3693.0          11.5          70    usa  
2  3436.0          11.0          70    

C:\Users\Sachit\AppData\Local\Temp\ipykernel_42548\1999747099.py:34: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
C:\Users\Sachit\AppData\Local\Temp\ipykernel_42548\1999747099.py:82: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_gu


Outlier Count:
mpg               0
cylinders         0
displacement      0
horsepower        4
weight            0
acceleration      2
model_year        0
weight_kg         0
vehicle_age       0
origin_encoded    0
dtype: int64

Outlier Rows:
                         name   mpg  cylinders  displacement  horsepower  \
6            chevrolet impala  14.0        8.0         454.0       220.0   
13    buick estate wagon (sw)  14.0        8.0         455.0       225.0   
95   buick electra 225 custom  12.0        8.0         455.0       225.0   
116        pontiac grand prix  16.0        8.0         400.0       230.0   
299               peugeot 504  27.2        4.0         141.0        71.0   
394                 vw pickup  44.0        4.0          97.0        52.0   

     weight  acceleration  model_year  origin    weight_kg mpg_category  \
6    2797.5           9.0          70     usa  1268.923620          Low   
13   3086.0          10.0          70     usa  1399.784912          Low  